σ-timed OCO breakouts on XAUUSD

Q1 (precondition): Does a run of small candles on the 15- or 30-minute
chart predict a larger move over the next H minutes than the HAR
volatility model already forecasts? 'Small' is measured relative to σ,
so the pattern means the same in calm and volatile years.

Q2 (strategy): When the HAR model and/or the small-candle pattern
indicate a quiet market expected to expand, do symmetric stop-entry
(OCO) breakout orders give better net outcomes than the same orders at
untimed squeezes or at random times?

Motivation: the owner's discretionary observation that compressed
periods of tiny candles tend to precede strong moves. This motivates the
hypothesis; it is not evidence for it.

Status: EXPLORATORY, Stage 1. Not a PREREG. Nothing here is evidence.
Q2 is only pursued if Q1 shows the pattern adds information.

## How this notebook works

- **Register first.** Every number, threshold, window, cost, seed or definition comes from the register cell below.
  Code reads a value only through `need(id)`, which raises unless the entry is **APPROVED**. A cell whose inputs aren't
  all approved prints what blocks it and computes nothing.
- **Guards.** Every frame is checked against the dev period with the repo's seal guards (`research/sealing.py`,
  `research/vol_features_r3/run_r3.py` `assert_sealed`). Nothing outside 2012-01-01 .. 2020-12-31 is read
  (2007–2011 excluded, decision log #16).
- **The work is the repo's code.** `stage1.py` (next to this notebook) only wires the repo's loader, `vol_block`
  and `sigma_compare`'s post-hoc scoring together, with values passed in from the register. Its tests:
  `research/squeeze_oco/test_stage1.py` (synthetic bars).
- **Data licence.** `XAUUSD.txt` is IBKR data, licensed for the owner's own use only. Outputs show aggregates and
  charts only, never a price row.
- **Record keeping.** The decision log below is append-only. `ATTEMPTS.json` counts outcome analyses; Stage 1 looks
  at no outcome and doesn't count.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
if not (HERE / "stage1.py").exists():                    # run from the repo root
    HERE = Path("research/squeeze_oco").resolve()
sys.path.insert(0, str(HERE))
import stage1  # noqa: E402

pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 80, "display.max_colwidth", 400)

## Register

`status`: **APPROVED** (usable), **APPROVED-BY-RULE** (usable: set by an approved rule from counts it cites), **PROPOSED** (a value with reasoning, waiting for the owner), **OPEN** (no value yet).
`source`: a repo file and line, or "owner, date". Changing an entry is a decision-log line.

In [ ]:
OWNER = "owner, 2026-10-26"
REGISTER = [
    # --- APPROVED: data, period, model (owner, with the repo source of each value) ---
    dict(id="DEV_START", item="dev period, first instant (UTC)", value="2012-01-01T00:00:00+00:00", status="APPROVED",
         source=f"{OWNER}: changed from 2007-01-01 (decision log #16). 2007-2011 excluded: a different feed/session convention "
                "(24 h, trading on holidays, Friday-evening bars), seen in the data check run on 2007-2020 at commit 338c180"),
    dict(id="DEV_END_EXCL", item="dev period, end, exclusive (UTC): data through 2020-12-31", value="2021-01-01T00:00:00+00:00",
         status="APPROVED", source=f"{OWNER} (unchanged by #16); research/vol_features_r3/run_r3.py:36 (CONF_END); research/sigma_compare/run_sigma.py:56"),
    dict(id="DATA_KEY", item="data file (manifest key)", value="database/ohlcv_data/XAUUSD.txt", status="APPROVED",
         source=f"{OWNER}; backend/datafeed/store/manifest.json:5"),
    dict(id="DATA_SHA256", item="XAUUSD.txt SHA-256 (manifest)", value="dc2d30cdee567d5043cd302ef9c4b87022d4145534fe0375dc116ae5c8dabfb9",
         status="APPROVED", source="backend/datafeed/store/manifest.json:26"),
    dict(id="LOADER", item="sealed research loader", value="research/vol_features_r3/run_r3.py load_conf", status="APPROVED",
         source=f"{OWNER} ('the repo's sealed research loader'); research/vol_features_r3/run_r3.py:89-99"),
    dict(id="CACHE_CHECK", item="the loader reads audit_long's cache, so the manifest check reaches the loaded rows only if "
         "that cache equals a fresh parse of XAUUSD.txt by the same code", value=True, status="APPROVED",
         source=f"{OWNER}: approved as the meaning of the manifest check (decision log #10); run_r3.py:90-93; "
                "research/vol_features_r2/audit_long.py:42-62"),
    dict(id="TZ", item="timestamps", value="UTC", status="APPROVED",
         source=f"{OWNER}; backend/datafeed/store/manifest.json:18; research/vol_features/DATA_AUDIT_LONG.md:11; run_r3.py:95"),
    dict(id="SIGMA_MODEL", item="HAR sigma model", value="parquet2021_2023", status="APPROVED",
         source=f"{OWNER}; backend/datafeed/features/vol_block.py:336; backend/datafeed/features/DECISIONS.md:69"),
    dict(id="SIGMA_MODEL_SHA256", item="its pinned SHA-256", value="dc0eb3c5fd31f1ee8f6e7a534b16930fbec453833aec9872b606afe963910dde",
         status="APPROVED", source="backend/datafeed/features/vol_block.py:337"),
    dict(id="HORIZONS", item="horizons H (minutes)", value=(60, 240), status="APPROVED",
         source=f"{OWNER}; backend/datafeed/features/vol_block.py:37 (only 60 and 240 fitted); research/sigma_compare/run_sigma.py:38"),
    dict(id="LICENCE", item="what committed outputs may show", value="aggregates and charts only; no raw price rows",
         status="APPROVED", source=f"{OWNER}; backend/datafeed/store/manifest.json:9"),
    # --- APPROVED: what Stage 1 describes and how calibration is scored (owner's task text) ---
    dict(id="DESCRIBE_COLUMNS", item="what step 3 describes", value=("sigma_px_60", "sigma_px_240", "sigma_ret_60", "sigma_ret_240", "L", "S_hd", "T"),
         status="APPROVED", source=f"{OWNER} (Stage 1 item 3)"),
    dict(id="DESCRIBE_GROUPS", item="how step 3 groups", value=("all", "year", "hour"), status="APPROVED",
         source=f"{OWNER} ('quantiles, by year, by hour of day')"),
    dict(id="CALIB_METHOD", item="calibration of the frozen model", value="sigma_compare post-hoc P2: mean RV/h and RMS z "
         "(and QLIKE) of HAR's h on valid, finite, non-crossing windows (posthoc.rows_from, posthoc.calib)", status="APPROVED",
         source=f"{OWNER} ('with the method from the sigma_compare study'); research/sigma_compare/POSTHOC.md:50-56; "
                "research/sigma_compare/posthoc.py:46-72"),
    dict(id="CALIB_COVERAGE", item="a window's RV counts only with >= this share of its H returns (part of CALIB_METHOD)",
         value=0.90, status="APPROVED", source="research/sigma_compare/run_sigma.py:39 (COVERAGE), used by build_context (:178)"),
    dict(id="CALIB_GROUPS", item="calibration reported", value=("all", "year"), status="APPROVED", source=f"{OWNER} ('overall and by year')"),
    # --- APPROVED 2026-10-26: Stage 1 grid and description (were PROPOSED) ---
    dict(id="EVAL_GRID", item="evaluation grid", value="every hourly label of vol_block.transform in the dev period (the close "
         "time of each 1-hour row, on the hour, UTC) that has a state by vol_block.state_at's rule (the first 120 labels of the "
         "loaded data have none); step 4 uses the subset whose H-window is valid and doesn't cross a close", status="APPROVED",
         source=f"{OWNER}: approved as proposed (decision log #7). Reasoning: HAR's state exists only at hourly labels (vol_block.py:25-31) and the model was fitted and scored on "
                "exactly this grid (run_sigma.py:149-190, posthoc.py:46-62); any finer grid would repeat the same state. "
                "Warm-up: vol_block.py:280"),
    dict(id="DESCRIBE_QUANTILES", item="quantiles reported in step 3", value=(0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99),
         status="APPROVED", source=f"{OWNER}: approved (decision log #8). Reasoning: the median and quartiles for the body, 5/95 and 1/99 % for the tails that "
                                   "a squeeze/expansion study is about; no value in the repo to reuse"),
    dict(id="DESCRIBE_HOUR", item="'hour of day' in step 3", value="UTC hour of the label (the end of the hour the state describes)",
         status="APPROVED", source=f"{OWNER}: approved, with a note in the output that sessions shift by one UTC hour with DST (decision log #9). Reasoning: TZ is UTC (register); the label is the time the state is known (vol_block.py:25-31)"),
    # --- OPEN / PROPOSED: Stage 1 ---
    dict(id="L_TERCILES", item="L tercile cut-offs (q1, q2): calm L < q1 <= normal < q2 <= stressed",
         value=(-0.4144399979178456, -0.018438540066194775), status="APPROVED",
         source=f"{OWNER}: approved as PR #42's candidate (decision log #19). feat/sigma-now:"
                "backend/datafeed/features/models/l_terciles_parquet2021_2023_v1.json:6-7 (q1, q2), :9 (SHA-256 6ad57b6d...), "
                ":10-11 (HAR training window 2021-01-03 23:00 .. 2023-12-29 21:59 UTC), :5 (17,711 labels). Not used by any Stage 1 cell"),
    # --- data check before steps 2-4 (owner, 2026-10-26) ---
    dict(id="DATA_CHECK", item="per year: where the daily break falls in UTC; bars on 1 Jan, 25 Dec, Good Friday; median "
         "bars per trading day", value="trade date = audit_long.session_date (18:00 New York opens the next date); season = "
         "audit_long.us_dst; break = minutes from a Mon-Thu trade date's last bar to the next trade date's first bar (its mode "
         "and share), and the modal UTC clock time of each trade date's last and first bar; holidays by UTC calendar date, "
         "Good Friday from pandas.tseries.holiday.GoodFriday; median bars over Mon-Fri trade dates", status="APPROVED",
         source=f"{OWNER} (the check's content, decision log #11; its measures, #13, confirmed by the owner, #17); "
                "research/vol_features_r2/audit_long.py:64-74"),
    dict(id="DATA_CHECK_ACCEPTED", item="the owner's decision on the data check's result (how 2007-2011, 2012-2014 and "
         "2015-2020 are treated) before steps 2-4", value="2007-2011 excluded (DEV_START = 2012-01-01); the one-minute "
         "break difference between 2012-2014 and 2015-2020 accepted as cosmetic", status="APPROVED",
         source=f"{OWNER} (decision log #16)"),
    # --- Q1 (owner, 2026-10-26: decision log #21-#26) ---
    dict(id="Q1_TIMEFRAME", item="candle timeframe (minutes)", value=15, status="APPROVED", source=f"{OWNER} (decision log #21)"),
    dict(id="Q1_SMALL_DEF", item="a 'small' candle: range relative to sigma",
         value=dict(option="B", range="high - low", level_H=60,
                    sigma_tf="sigma_tf^2 = var_60 x P_candle / P_60 = P_candle x exp(beta_60 . [1, L, S_hd, T]) x c_60, "
                             "state of the latest hourly label at or before the candle's start (vol_block.state_at); "
                             "price units: x close of the last bar before the start",
                    score="range / sigma_tf_px"),
         status="APPROVED", source=f"{OWNER}: option B, range = high - low (decision log #21); PROPOSALS.md option B; "
                                   "the anchor and formula are Claude's reading, owner to confirm (decision log #27, Q1_SETUP.md section 2.1)"),
    dict(id="Q1_THRESHOLD_KIND", item="how 'small' is thresholded",
         value=dict(kind="quantile per hour", hour_tz="America/New_York", hour_of="the candle's start (a run's candles share one NY hour)",
                    rule="score <= the q-quantile (numpy linear) of the scores of the complete scored candles starting at the same "
                         "New York hour in the training years",
                    walk_forward="by calendar year, expanding from DEV_START; year Y uses DEV_START's year .. Y-1, never Y; a run uses its label's year"),
         status="APPROVED", source=f"{OWNER}: changed before any outcome (decision log #31). Reason: with one pooled quantile (decision log #22) "
                                   "the runs concentrated at 19/20/23/00 UTC, a time-of-day artefact around the daily break "
                                   "(Q1_SETUP.md section 4.1); per NY hour, so the hour follows the sessions through DST. "
                                   "Expanding as research/sigma_compare/run_sigma.py:227"),
    dict(id="Q1_RUN_ANCHOR", item="where a run ends", value="at an hourly label of EVAL_GRID; its r candles are the last r x Q1_TIMEFRAME "
         "minutes of the hour before the label", status="APPROVED", source=f"{OWNER} ('Runs end at an hourly label and cover the hour before it', decision log #23)"),
    dict(id="Q1_RULE", item="the rule that picks Q1_SMALL_Q and Q1_RUN_LENGTH, decided before any counts",
         value=dict(r_order=(4, 3), q_order=(0.20, 0.25), min_runs=300, span=("2012-01-01", "2021-01-01"),
                    runs="non-overlapping (sharing no candle) runs at EVAL_GRID labels", if_none="stop and report: too little data"),
         status="APPROVED", source=f"{OWNER} (decision log #23); 'non-overlapping' read as sharing no candle (#27)"),
    dict(id="Q1_SMALL_Q", item="q: the share of candles counted as small", value=0.20, status="APPROVED-BY-RULE",
         source="Q1_RULE with the per-hour threshold (decision log #32, #37): r = 4 first; q = 0.20 is the strictest and gives 349 >= 300 "
                "runs at grid labels in 2012-2020 (results/q1_run_counts.json: totals.'q=0.2,r=4'.runs = 349, chosen_by_rule; all in "
                "2013-2020). The other readings choose the same (for_comparison_only). Supersedes #28 (pooled: 510)"),
    dict(id="Q1_RUN_LENGTH", item="r: candles in a run", value=4, status="APPROVED-BY-RULE",
         source="Q1_RULE (decision log #37): as Q1_SMALL_Q; results/q1_run_counts.json chosen_by_rule = {q 0.2, r 4, n 349}"),
    dict(id="Q1_SENSITIVITY", item="the q values Q1_RULE doesn't choose", value="reported as descriptive sensitivity only, never tested",
         status="APPROVED", source=f"{OWNER} (decision log #23)"),
    dict(id="Q1_H", item="horizon H for Q1", value=dict(primary=240, secondary=60, multiplicity="BH across the two"),
         status="APPROVED", source=f"{OWNER} (decision log #24); HORIZONS"),
    dict(id="Q1_SCORING", item="how Q1 is scored against HAR",
         value=dict(test="A: sigma_compare's method. QLIKE of HAR's h against HAR + the pattern indicator on the same valid, "
                         "non-crossing rows; walk-forward folds with an embargo; Diebold-Mariano with Newey-West lag "
                         "Q1_NW_LAG_240 / Q1_NW_LAG_60; BH across the two H (q as BH_Q)",
                    descriptive="B: conditional calibration (mean RV/h, RMS z) on pattern rows vs the others; no test",
                    not_used="C"),
         status="APPROVED", source=f"{OWNER} (decision log #18, #25); research/sigma_compare/run_sigma.py:39-46, 207-222, 402-428"),
    dict(id="Q1_NW_LAG_240", item="Newey-West lag (rows) of the DM test at H = 240", value=8, status="APPROVED",
         source=f"{OWNER}: approved as proposed (decision log #33); reasoning Q1_SETUP.md section 5"),
    dict(id="Q1_NW_LAG_60", item="Newey-West lag (rows) of the DM test at H = 60", value=8, status="APPROVED",
         source=f"{OWNER}: approved as proposed (decision log #33); reasoning Q1_SETUP.md section 5"),
    dict(id="Q1_FOLDS", item="the walk-forward folds of the Q1 test",
         value=dict(test_blocks="[Y-01-01, Y+1-01-01) UTC for Y = 2014 .. 2020 (7 folds), pooled over the test rows per H",
                    training_rows="labels t with 2013-01-01 <= t <= s_k - 24 h and window end E <= s_k (expanding)",
                    embargo="24 h", first_training_label="2013-01-01: 2012 has no threshold, so no pattern indicator"),
         status="APPROVED", source=f"{OWNER}: expanding window, test year by year 2014-2020, 24 h embargo (decision log #34); exact edges "
                                   "approved as proposed (#38; Q1_SETUP.md section 9), following research/sigma_compare/PREREG.md:95-97"),
    dict(id="Q1_AUG_FIT", item="what 'HAR + the pattern indicator' fits per fold",
         value="the pinned HAR (VolParams, beta, c) fixed; per fold and H only the pattern indicator's coefficient gamma, on the fold's "
               "training rows: h_aug = h_HAR x exp(gamma x I_run)", status="APPROVED", source=f"{OWNER} (decision log #35)"),
    dict(id="Q1_AUG_ESTIMATOR", item="how gamma is estimated, and on which training rows",
         value=dict(gamma="ln(mean(RV / h_HAR)) over the training rows with I_run = 1: the QLIKE-minimising gamma given h_HAR",
                    rows="valid, finite, non-crossing training rows (the population the test scores)"),
         status="APPROVED", source=f"{OWNER}: approved as proposed (decision log #38); Q1_SETUP.md section 9"),
    dict(id="Q1_READING_RULE", item="how the Q1 result is read, fixed before any outcome",
         value=dict(primary="two-sided Diebold-Mariano, HAR vs HAR + pattern, H = 240; BH across H = 240 and H = 60 at q = BH_Q",
                    adds_information="the H = 240 test is significant after BH",
                    q2_proceeds="adds_information AND pooled gamma > 0 AND gamma > 0 in at least 5 of the 7 test years",
                    min_positive_years=5,
                    descriptive="the H = 60 result, the q = 0.25 sensitivity, conditional calibration, per-hour and per-year breakdowns"),
         status="APPROVED", source=f"{OWNER} (decision log #39)"),
    dict(id="Q1_GATE_GAMMA", item="which gamma the Q2 gate's 'pooled gamma' and 'gamma in a test year' mean",
         value=dict(H=240, pooled="Q1_AUG_ESTIMATOR on all test-year rows (2014-2020) with a run: out of sample",
                    per_year="Q1_AUG_ESTIMATOR on each test year's rows with a run (out of sample)",
                    also_reported="the fold gammas (fitted on each fold's training rows), and the gate read with them"),
         status="PROPOSED", source="Claude, 2026-10-26, written before the run (decision log #40): the rule's wording allows both "
                                   "per-test-year readings; the gate is reported under both"),
    dict(id="BH_Q", item="BH false-discovery level", value=0.05, status="APPROVED",
         source=f"{OWNER}: Q1_SCORING approved 'sigma_compare's method' with 'BH q 0.05' (decision log #18); research/sigma_compare/run_sigma.py:42"),
    # --- OPEN: Q2 ---
    *[dict(id=i, item=t, value=None, status="OPEN", source=f"{OWNER}") for i, t in [
        ("Q2_SQZ_LOOKBACK", "squeeze lookback"), ("Q2_SQZ_THRESHOLD", "squeeze threshold"),
        ("Q2_EXPANSION", "expansion condition"), ("Q2_ENTRY_BUFFER_SIGMA", "entry buffer in sigma"),
        ("Q2_ARMING_WINDOW", "arming window"), ("Q2_STOP_RULE", "stop rule"), ("Q2_TARGET_MULT", "target multiple"),
        ("Q2_TIME_CAP", "time cap"), ("Q2_BOTH_FILLED", "both-sides-filled handling"), ("Q2_SPREAD", "spread"),
        ("Q2_STOP_SLIPPAGE", "stop-fill slippage"), ("Q2_COMMISSION", "commission"), ("Q2_NEWS", "news handling"),
        ("Q2_BENCHMARKS", "benchmarks"), ("Q2_MAX_GRID", "maximum grid size"), ("Q2_SEED", "random seed")]],
]
_REG = {r["id"]: r for r in REGISTER}
assert len(_REG) == len(REGISTER), "duplicate register id"


USABLE = ("APPROVED", "APPROVED-BY-RULE")      # APPROVED-BY-RULE: set by an approved rule from cited counts (decision log #26)


class RegisterError(RuntimeError):
    pass


def need(id_):
    """The value of an APPROVED (or APPROVED-BY-RULE) register entry; raises otherwise."""
    if id_ not in _REG:
        raise RegisterError(f"{id_}: not in the register")
    r = _REG[id_]
    if r["status"] not in USABLE:
        raise RegisterError(f"{id_} is {r['status']}: {r['item']} (value {r['value']!r}; {r['source']})")
    return r["value"]


def blocked(*ids):
    """The ids among these that aren't APPROVED."""
    return [i for i in ids if _REG[i]["status"] not in USABLE]


reg = pd.DataFrame(REGISTER)[["id", "status", "item", "value", "source"]]
print(reg["status"].value_counts().to_string())
reg

## Context: what the repo's reports say (every claim CITED: file:line, checked 2026-10-26)

**The vol block (L, S_hd, T).**
- CITED: L, S and T beat the 13 earlier volatility features at forecasting deseasonalized future variance, on XAUUSD
  2007–2020: out-of-sample R² 0.73 vs 0.52 (4 h) and 0.80 vs 0.55 (24 h), 24/24 half-years
  (`research/vol_block/VOL_BLOCK_DECISION.md:11-15`). The signs L +, S +, T − hold in every fold
  (`VOL_BLOCK_DECISION.md:16-17`).
- CITED: **S_hd** (S minus its training-window hour mean) is an engineering decision; its out-of-sample effect was only
  measured exploratorily (`VOL_BLOCK_DECISION.md:31-34`). R2 (no clock) failed narrowly in round 3
  (`VOL_BLOCK_DECISION.md:23-27`).

**HAR vs the plug-in (sigma_compare).**
- CITED: on CONF 2007–2020, non-crossing windows: QLIKE 0.2493 vs 0.3044 at 60 min (n 68,040) and 0.1838 vs 0.2190 at
  240 min (n 58,920) (`research/sigma_compare/REPORT.md:17-18`): −18 % and −16 % (`REPORT.md:20`), better in
  24/24 folds (`REPORT.md:22`).
- CITED + INFERRED: **this notebook's dev period (2012–2020) lies inside that study's CONF period (2007–2020).** The HAR *form* was chosen on it, although the pinned
  model's coefficients and VolParams were fitted on 2021–2023 only
  (`backend/datafeed/features/models/sigma_har_parquet2021_2023_v1.json:657-658`: train 2021-01-03 .. 2023-12-29).
  So the dev period is seen data for the method, unseen for this model's fit.

**Out of era.**
- CITED: frozen on `XAUUSD.txt` 2007–2020 and scored without refit on the parquet 2021–2023: QLIKE 0.2142 vs 0.2722
  (60 min, n 16,921) and 0.1601 vs 0.1987 (240 min, n 14,608), −21 % and −19 %
  (`research/sigma_compare/POSTHOC_RESULT.md:18-21`).

**HAR calibration.**
- CITED: walk-forward on DEV 2022–2023, the parquet recipe (the pinned model's, "SHORT"): mean RV/h 0.994 (60) and 0.982
  (240), RMS z 0.968 and 0.969 (`POSTHOC_RESULT.md:36,38`), inside the [0.9, 1.1] band (`POSTHOC.md:60`).
- CITED: **a difference to note:** the post-hoc rule favoured the txt fit (LONG) as the default
  (`POSTHOC_RESULT.md:8-9,41`). The owner later made `parquet2021_2023` the default, on QLIKE
  (`backend/datafeed/features/DECISIONS.md:69-76`). That is the model this notebook uses (register SIGMA_MODEL).
- INFERRED (from a search of research/ and docs/): the calibration of this model on 2007–2020 (step 4) has not been reported anywhere yet.

**Relevant nulls.**
- CITED: **levels (2007–2020 CONF):** 0 of 6 hypotheses supported; daily-level breaks continued *less* than placebo
  breaks, −7.3 pp [−9.6, −5.0] (`research/levels/REPORT.md:19`).
- CITED: **levels_confirm (2024-01-01 .. 2026-01-30):** REJECTED; near-open PDH/PDL breaks continued *more* than
  distance-matched placebos, +11.3 pp [+4.0, +18.5] (`research/levels_confirm/REPORT.md:10,18`). INFERRED: the two
  break studies disagree in sign across eras, and neither supports a usable rule.
- **Asia → London continuation: not established (legacy notebook cell only).** CITED: there is no study or report in the repo. The only analysis is an exploratory
  cell of the legacy notebook, `legacy/notebooks/backtest.ipynb` (cell 16, 2022-01-01 .. 2026-01-30 per cell 3):
  its saved output reads "Overall continuation rate: 52.3 % (685 / 1311 days)", with filtered subsets at
  55.7–58.8 % and no test, no PREREG and no placebo. INFERRED: it is not a null result in the repo's sense.
- No numbers were quoted in the request, so there were none to check against the reports.

**What HAR does and doesn't say.** CITED: HAR forecasts the *variance* of the log return over the next H market-open
minutes, `var(t, H) = P(t, H) × exp(β · [1, L, S_hd, T]) × c` (`backend/datafeed/features/vol_block.py:33`).
INFERRED from that definition: **it predicts the size of moves, not their direction, and not whether a move persists.**
INFERRED (from a search of research/): no study in the repo tested direction or persistence from it.

## Decision log (append-only)

| # | date | id | decision | by |
|---|---|---|---|---|
| 1 | 2026-10-26 | (all) | Notebook, register rule, Q1/Q2 framing, the APPROVED items DEV_START, DEV_END_EXCL, DATA_KEY, LOADER, TZ, SIGMA_MODEL, HORIZONS, LICENCE, DESCRIBE_COLUMNS, DESCRIBE_GROUPS, CALIB_METHOD, CALIB_GROUPS, and the OPEN list | owner |
| 2 | 2026-10-26 | (branch) | `research/squeeze-oco` is stacked on `feat/data-manifest` (PR #41): the manifest check exists only there | Claude |
| 3 | 2026-10-26 | CACHE_CHECK | "Through the manifest check" read as: verify XAUUSD.txt's SHA-256, and check that the loader's cache equals a fresh parse of it by the loader's own code, since `load_conf` reads the cache, not the file. Owner to confirm | Claude |
| 4 | 2026-10-26 | CALIB_COVERAGE, DATA_SHA256, SIGMA_MODEL_SHA256 | Taken as part of the approved items they belong to (sigma_compare's method; the manifest; the pinned model), with their file:line | Claude |
| 5 | 2026-10-26 | EVAL_GRID | Found while testing: over a whole frame, `vol_block.transform` gives finite L, S_hd, T for the first 120 hourly labels, whose 120-hour window reaches the first (possibly partial) row; `state_at` refuses them. The grid proposal follows `state_at` (`stage1.features`) | Claude |
| 6 | 2026-10-26 | (context) | Asia → London continuation: no report in the repo; only the legacy notebook cell, cited as such | Claude |
| 7 | 2026-10-26 | EVAL_GRID | APPROVED as proposed | owner |
| 8 | 2026-10-26 | DESCRIBE_QUANTILES | APPROVED: 1, 5, 25, 50, 75, 95, 99 % | owner |
| 9 | 2026-10-26 | DESCRIBE_HOUR | APPROVED: the UTC hour; the output notes that sessions shift by one UTC hour with DST | owner |
| 10 | 2026-10-26 | CACHE_CHECK | "The loader's cache equals a fresh parse" APPROVED as the meaning of the manifest check | owner |
| 11 | 2026-10-26 | DATA_CHECK | Before steps 2–4: per year 2007–2020, where the daily break falls in UTC, bars on CME holidays (1 Jan, 25 Dec, Good Friday), median bars per trading day. If 2007–2010 differ from 2011+ (feed, timezone or calendar): report and stop, adjust nothing | owner |
| 12 | 2026-10-26 | L_TERCILES, (context) | L_TERCILES stays OPEN. Asia → London cited as "not established (legacy notebook cell only)" | owner |
| 13 | 2026-10-26 | DATA_CHECK | The check's measures written from the repo's definitions (audit_long.session_date, us_dst; pandas' Good Friday); the break measured as the gap to the next trade date's first bar, since the first bar at 18:00 New York follows from the trade-date cut itself. Owner to confirm | Claude |
| 14 | 2026-10-26 | DATA_CHECK_ACCEPTED | The check shows three regimes (see "Data check: result"). By decision #11, steps 2–4 are stopped; DATA_CHECK_ACCEPTED added as OPEN | Claude |
| 16 | 2026-10-26 | DATA_CHECK_ACCEPTED, DEV_START | Dev period for this study = 2012-01-01 .. 2020-12-31. 2007–2011 excluded (different feed/session convention: 24 h, holidays, Friday-evening bars). The 2012–2014 one-minute break difference accepted as cosmetic. Register and guard updated | owner |
| 17 | 2026-10-26 | DATA_CHECK | The check's measures (#13) confirmed | owner |
| 18 | 2026-10-26 | Q1_SCORING | APPROVED (sigma_compare's method). The other Q1 items wait for the owner's review of PROPOSALS.md | owner |
| 19 | 2026-10-26 | L_TERCILES | APPROVED as PR #42's candidate (HAR training window 2021-01-03 .. 2023-12-29) | owner |
| 20 | 2026-10-26 | (data check) | With DEV_START = 2012-01-01, the guard refuses 2007–2011, so the notebook can no longer show the 2007–2020 check. Its record is the executed notebook at commit 338c180 (cited, not rerun); the check cells now run on the new dev period | Claude |
| 21 | 2026-10-26 | Q1_TIMEFRAME, Q1_SMALL_DEF | 15 min; option B (range / σ_tf, σ_tf from the model's own seasonal mass for that slot), range = high − low | owner |
| 22 | 2026-10-26 | Q1_THRESHOLD_KIND | Quantile (the smallest q share of scores), fitted on training years only, walk-forward; never on the year being scored | owner |
| 23 | 2026-10-26 | Q1_RULE, Q1_RUN_ANCHOR, Q1_SENSITIVITY | (q, r) by a rule decided before any counts: r = 4; q = the strictest of 0.20 and 0.25 giving ≥ 300 non-overlapping runs in 2012–2020; else r = 3 the same way; else stop (too little data). Non-chosen q descriptive only. Runs end at an hourly label and cover the hour before it | owner |
| 24 | 2026-10-26 | Q1_H | 240 primary, 60 secondary, BH across the two | owner |
| 25 | 2026-10-26 | Q1_SCORING | A as the test; B (conditional calibration on pattern rows) descriptive only, no test; C not used | owner |
| 26 | 2026-10-26 | Q1_SMALL_Q, Q1_RUN_LENGTH | The rule's result is recorded as APPROVED-BY-RULE, citing the counts; `need()` accepts that status | owner |
| 27 | 2026-10-26 | Q1_SMALL_DEF, Q1_THRESHOLD_KIND, Q1_RULE | Claude's readings, committed before any count (765397a, `Q1_SETUP.md` section 2): σ_tf² = var_60 × P_candle / P_60 from the state at the candle's start; threshold over all complete scored candles of the training years, numpy linear quantile, score ≤ threshold, expanding by calendar year (so 2012 has no threshold and no runs); a run uses its label's year; "non-overlapping" = sharing no candle (true by construction at hourly labels); r = 3 covers the last 45 min; runs counted at EVAL_GRID labels. Owner to confirm | Claude |
| 28 | 2026-10-26 | Q1_SMALL_Q, Q1_RUN_LENGTH | Set by Q1_RULE from `results/q1_run_counts.json` (see "Q1 setup") | rule |
| 29 | 2026-10-26 | Q1_NW_LAG_240, Q1_NW_LAG_60, Q1_FOLDS, Q1_AUG_FIT, BH_Q | NW lags PROPOSED (`Q1_SETUP.md` section 5); fold edges and the augmented fit added as OPEN; BH_Q entered as part of the approved Q1_SCORING | Claude |
| 30 | 2026-10-26 | (readings) | Q1_SETUP.md section 2 confirmed (non-overlapping = sharing no candle; no threshold in 2012) | owner |
| 31 | 2026-10-26 | Q1_THRESHOLD_KIND | Changed before any outcome: "small" = score below the q-quantile of scores at the same hour, hour in America/New_York time, fitted on past years only, walk-forward. Reason: with the pooled quantile the runs concentrated at 19/20/23/00 UTC, a time-of-day artefact around the daily break | owner |
| 32 | 2026-10-26 | Q1_SMALL_Q, Q1_RUN_LENGTH | Recount on 2012–2020 with the per-hour threshold and apply the same rule. #28's result (pooled) is superseded; back to OPEN until the rule sets them | owner |
| 33 | 2026-10-26 | Q1_NW_LAG_240, Q1_NW_LAG_60 | 8 at both H, APPROVED | owner |
| 34 | 2026-10-26 | Q1_FOLDS | Expanding window, test year by year 2014–2020, 24 h embargo; exact edges to be PROPOSED | owner |
| 35 | 2026-10-26 | Q1_AUG_FIT | Pinned HAR β fixed; per fold fit only the pattern indicator's coefficient on past data | owner |
| 36 | 2026-10-26 | Q1_FOLDS, Q1_AUG_ESTIMATOR | Edges PROPOSED; how γ is estimated added as PROPOSED (`Q1_SETUP.md` section 9) | Claude |
| 37 | 2026-10-26 | Q1_SMALL_Q, Q1_RUN_LENGTH | Set by Q1_RULE from the recount (`results/q1_run_counts.json`; see "Q1 setup") | rule |
| 38 | 2026-10-26 | Q1_FOLDS, Q1_AUG_ESTIMATOR | APPROVED as proposed. Sample accepted as is (rule met as written; no change) | owner |
| 39 | 2026-10-26 | Q1_READING_RULE | Fixed before any outcome: primary = two-sided DM, HAR vs HAR + pattern, H = 240, BH across H = 240 and 60 at q = 0.05. "Pattern adds information" if H = 240 is significant after BH. Q2 proceeds only if that holds AND pooled γ > 0 AND γ > 0 in ≥ 5 of the 7 test years. Everything else descriptive | owner |
| 40 | 2026-10-26 | Q1_GATE_GAMMA | Claude's reading, before the run: the gate's γ at H = 240; "pooled" and "per test year" = the estimator on the test rows with a run (out of sample). The gate is also reported with the fold γ (fitted on past rows). PROPOSED, owner to confirm | Claude |
| 41 | 2026-10-26 | (ATTEMPTS) | Q1 run once: `ATTEMPTS.json` 0 → 1 before any RV was read (`q1_test_run.py`) | owner / Claude |
| 15 | 2026-10-26 | (all) | Overnight rules: never fill an OPEN item or promote a PROPOSED one; label every claim OBSERVED, CITED or INFERRED (UNVERIFIED if it can't be checked); stop Stage 1 if the data check fails and continue with the safe tasks | owner |


In [ ]:
a = stage1.attempts()
print(f"outcome analyses so far: {a['outcome_analyses']}  (Stage 1 is descriptive and doesn't count)")

## Guard: the dev period only

`stage1.dev_guard` refuses any timestamp outside [DEV_START, DEV_END_EXCL), now 2012-01-01 .. 2020-12-31 (decision log #16). The upper side is the repo's
guards: `research/sealing.py` `assert_before_global_seal` (nothing from 2026-10-01) and `run_r3.assert_sealed`
with the study end. The repo has no lower-side guard, so that side is one comparison. Every frame below goes
through it; this cell shows that it refuses.

In [ ]:
DEV_START, DEV_END = pd.Timestamp(need("DEV_START")), pd.Timestamp(need("DEV_END_EXCL"))
for bad in ["2007-01-01 00:00", "2011-12-31 23:59", "2021-01-01 00:00", "2026-10-01 00:00"]:
    try:
        stage1.dev_guard(pd.DatetimeIndex([bad], tz="UTC"), DEV_START, DEV_END, "probe")
        raise SystemExit(f"guard let {bad} through")
    except AssertionError as e:
        print(f"refused {bad}: {str(e)[:90]}")
stage1.dev_guard(pd.DatetimeIndex(["2012-01-01 00:00", "2020-12-31 23:59"], tz="UTC"), DEV_START, DEV_END, "probe")
print("accepted 2012-01-01 00:00 and 2020-12-31 23:59")

## Stage 1, step 1: load the dev-period bars

The data guard (`research/data_guard.py`), the manifest's SHA-256 of `XAUUSD.txt`, the cache check, then
`run_r3.load_conf` for [DEV_START, DEV_END_EXCL), through the guard. The model is loaded with its SHA-256 checked.
Shown: aggregates per year only.

In [ ]:
assert need("TZ") == "UTC" and need("LOADER") == "research/vol_features_r3/run_r3.py load_conf"
m, checks = stage1.load_dev(need("DATA_KEY"), DEV_START, DEV_END, need("CACHE_CHECK"))
assert checks["manifest_sha256"] == need("DATA_SHA256")
model = stage1.load_model(need("SIGMA_MODEL"), need("SIGMA_MODEL_SHA256"))
print(checks)
print(f"model {model.name}: SHA-256 {model.sha256} checked; horizons {sorted(model.horizons)}")
assert tuple(sorted(model.horizons)) == tuple(need("HORIZONS"))
print(f"bars: {len(m):,}, {m.index.min()} .. {m.index.max()} (UTC)")
cov = stage1.coverage_table(m)
cov

## Stage 1, data check (before steps 2–4)

Run first on 2007–2020 (commit 338c180: 2007–2011 differ, Stage 1 stopped, decision log #14). The owner then
excluded 2007–2011 (#16). The same check now runs on the dev period 2012–2020 (register DATA_CHECK): where the
daily break falls in UTC, bars on 1 January, 25 December and Good Friday, and the median bars per trading day.

In [ ]:
assert need("DATA_CHECK")
brk = stage1.break_table(m)
brk.round(3)

In [ ]:
hol = stage1.holiday_table(m, range(DEV_START.year, DEV_END.year))
hol

In [ ]:
bpd = stage1.bars_per_trade_date(m)
bpd

## Data check: result

- **CITED, on 2007–2020:** the first run (executed notebook at commit `338c180`, "Data check: result") found that
  2007–2011 differ from 2012+: no daily break, bars on 1 Jan / 25 Dec / Good Friday at all hours, and
  Friday-evening sessions. Stage 1 stopped there (decision log #14).
- **The owner excluded 2007–2011** (decision log #16). The 2012–2014 one-minute break difference is accepted as
  cosmetic.
- **OBSERVED, on the dev period 2012–2020 (the tables above):**
  - Every year has a daily break: its most common gap is 60 min in 2012–2014 and 61 min in 2015–2020.
  - The next session's first bar is at 22:00 UTC in US summer time and 23:00 UTC in winter (shares 0.859–1.000).
  - Holiday bars occur only in the 23:00 UTC hour (0–120 a year); Good Friday has 0 bars every year; no year has
    a bar on a UTC Saturday.
  - The median bars per Mon–Fri trade date is 1,377–1,380.
  - Weekend trade dates: 0, except 1 in 2014.
- **INFERRED:** within the dev period, the calendar and session convention are the same from year to year, apart
  from the accepted one-minute difference. Steps 2–4 run below.


## Stage 1, steps 2–3: the vol state on the evaluation grid, described

Runs only once DATA_CHECK_ACCEPTED, EVAL_GRID, DESCRIBE_QUANTILES and DESCRIBE_HOUR are approved. The hour of day is
the UTC hour of the label; sessions shift by one UTC hour with US DST.

In [ ]:
b = blocked("DATA_CHECK_ACCEPTED", "EVAL_GRID", "DESCRIBE_QUANTILES", "DESCRIBE_HOUR", "DESCRIBE_COLUMNS", "DESCRIBE_GROUPS")
if b:
    print("BLOCKED by:", ", ".join(f"{i} ({_REG[i]['status']})" for i in b))
    tbl = None
else:
    ctx = stage1.context(m, DEV_END, need("HORIZONS"), need("CALIB_COVERAGE"))
    tbl = stage1.state_table(ctx, model, DEV_START, DEV_END)
    cols, q = list(need("DESCRIBE_COLUMNS")), list(need("DESCRIBE_QUANTILES"))
    assert need("DESCRIBE_HOUR").startswith("UTC hour")
    print("Note: the hour of day is the UTC hour. Sessions (and the daily break) shift by one UTC hour with US DST.")
    print(f"grid: {len(tbl):,} hourly labels with a state")
    for g in need("DESCRIBE_GROUPS"):
        if g == "all":
            print("--- all")
            display(stage1.describe(tbl, cols, q, None))
            continue
        for col in cols:
            print(f"--- {col} by {g}" + (" (UTC hour of the label; sessions shift by one UTC hour with US DST)" if g == "hour" else ""))
            display(stage1.describe(tbl, [col], q, g)[col])

## Stage 1, step 4: calibration on the dev period (the pinned model never saw it)

sigma_compare's post-hoc P2 metrics (mean RV/h, RMS z, QLIKE) of the frozen model on valid, non-crossing
windows, overall and by year. Runs only once DATA_CHECK_ACCEPTED and EVAL_GRID are approved.

In [ ]:
b = blocked("DATA_CHECK_ACCEPTED", "EVAL_GRID", "CALIB_METHOD", "CALIB_COVERAGE", "CALIB_GROUPS")
if b:
    print("BLOCKED by:", ", ".join(f"{i} ({_REG[i]['status']})" for i in b))
else:
    assert need("CALIB_GROUPS") == ("all", "year")
    cal = stage1.calibration(ctx, model, DEV_START, DEV_END, True)
    display(cal)

## What Stage 1 found (run 2026-10-26, dev period 2012–2020)

**Data (step 1, data check).**
- OBSERVED: `XAUUSD.txt` matches its manifest SHA-256 (`dc2d30cd…`), and the loader's cache equals a fresh parse
  (6,971,972 rows before 2024).
- OBSERVED: the dev period holds 3,180,150 1-min bars, 2012-01-02 23:00 .. 2020-12-31 21:59 UTC (step 1 output).
  Within it, the session and calendar convention is consistent (data check, above).
- OBSERVED: the guard refuses 2007-01-01, 2011-12-31 23:59, 2021-01-01 and 2026-10-01, and accepts the period's
  edges.

**The vol state (steps 2–3).**
- OBSERVED: **52,929** hourly labels with a state. The hour of day below is the UTC hour of the label. Sessions,
  and the daily break, shift by one UTC hour with US DST.
- OBSERVED, overall medians and [1 %, 99 %] quantiles:
  - σ over 60 min: $2.05 [0.79, 8.26], i.e. 0.151 % [0.061, 0.530];
  - σ over 240 min: $4.44 [2.03, 15.56], i.e. 0.326 % [0.157, 0.986];
  - L −0.195 [−1.52, 2.20], S_hd −0.089 [−1.56, 1.87], T −0.079 [−1.00, 0.92].
- OBSERVED, by year: the median σ_ret over 60 min is lowest in 2017–2019 (0.121 % each) and highest in 2020
  (0.198 %) and 2013 (0.194 %). The median L ranges from −0.761 (2019) to +0.433 (2013) and +0.376 (2020).
- OBSERVED, by hour: the median σ_ret over 60 min peaks for windows starting at 13:00 UTC (0.297 %), 14:00
  (0.269 %) and 12:00 (0.260 %). It is lowest at 04:00 (0.088 %) and 03:00 (0.091 %).
- INFERRED: the peak hours cover the US data releases and the US open (12:30–14:30 UTC, depending on DST).

**Calibration of the pinned model on 2012–2020, which it never saw (step 4: sigma_compare's P2 metrics,
valid non-crossing windows).**
- OBSERVED: H = 60: n 50,516, mean RV/h **1.037**, RMS z **0.969**, QLIKE 0.255. H = 240: n 43,619, mean RV/h
  **1.002**, RMS z **0.969**, QLIKE 0.199.
- OBSERVED, by year: mean RV/h ranges 0.952–1.135 (60 min) and 0.881–1.122 (240 min); RMS z ranges 0.917–1.021
  (60) and 0.902–1.054 (240).
  - 2015 is the highest at both horizons (1.135 / 1.122), and 2013 at 240 min is 1.117.
  - 2017 and 2018 at 240 min are the lowest (0.890 / 0.881).
- CITED: sigma_compare's "close to 1" band is [0.9, 1.1] (`research/sigma_compare/POSTHOC.md:60`).
- INFERRED: overall, all four numbers lie inside it. By year, mean RV/h leaves it in 2015 (both H), in 2013
  (240), and in 2017–2018 (240, below). Here the band is a reference, not a gate.
- INFERRED: HAR's variance forecast is about right on average over these 9 years. Year by year, realised variance
  runs up to 13.5 % above the forecast (2015, 60 min) and up to 11.9 % below it (2018, 240 min).

**Not done.** Q1: its parameters wait for the owner (PROPOSALS.md). Outcome analyses: 0 (`ATTEMPTS.json`).


## Q1 setup: the small-candle score and the event counts (descriptive, before any outcome)

Steps 1–3 of the owner's Q1 order (2026-10-26). Option B's score per complete 15-min candle, its walk-forward
quantile threshold, the runs at the hourly labels of the grid, and the rule that picks (q, r). **Counts only:**
no forward return, no RV, no outcome is read, and `ATTEMPTS.json` stays at 0. The readings behind the code are in
`Q1_SETUP.md` section 2 (committed before any count). The same numbers are written by `q1_runs.py` to
`results/q1_run_counts.json`.

In [ ]:
import q1
b = blocked("Q1_TIMEFRAME", "Q1_SMALL_DEF", "Q1_THRESHOLD_KIND", "Q1_RUN_ANCHOR", "Q1_RULE", "Q1_H", "EVAL_GRID", "DESCRIBE_QUANTILES")
assert not b, b
TF, SD, RULE = need("Q1_TIMEFRAME"), need("Q1_SMALL_DEF"), need("Q1_RULE")
assert SD["option"] == "B" and SD["range"] == "high - low"
YEARS = list(range(DEV_START.year, DEV_END.year))
sc = q1.candle_scores(m, ctx, model, TF, SD["level_H"])
stage1.dev_guard(sc.index, DEV_START, DEV_END, "Q1 candle scores")
print(f"complete {TF}-min candles: {len(sc):,}; with a score: {int(sc['score'].notna().sum()):,}; "
      f"with a closed minute: {int((sc['closed_minutes'] > 0).sum()):,}")
display(stage1.describe(sc.dropna(subset=['score']), ["score"], list(need("DESCRIBE_QUANTILES")), "year")["score"])
print("score by UTC hour of the candle's start (sessions shift by one UTC hour with US DST):")
display(stage1.describe(sc.dropna(subset=['score']), ["score"], list(need("DESCRIBE_QUANTILES")), "hour")["score"])

In [ ]:
H = need("Q1_H")
ec = q1.event_counts(sc, tbl[[f"crossing_{h}" for h in need("HORIZONS")]], YEARS, RULE["q_order"], RULE["r_order"], TF,
                     (H["primary"], H["secondary"]), tuple(need("DESCRIBE_QUANTILES")), need("Q1_THRESHOLD_KIND")["hour_tz"])
for q, t in ec["thresholds"].items():
    print(f"--- threshold q={q}: year scored x New York hour of the candle's start")
    display(pd.Series(t).unstack().round(3))
per = pd.concat({f"q={q}, r={r}": pd.DataFrame(v).T for (q, r), v in ec["per_year"].items()}, axis=1)
per.loc["total"] = per.sum()
per

In [ ]:
hours = pd.concat({f"q={q}, r={r}": pd.DataFrame(v).T for (q, r), v in ec["per_hour"].items()}, axis=1).fillna(0).astype(int)
hours.index.name = "New York hour of the run's candles"
hours.loc["total"] = hours.sum()
hours

In [ ]:
clu = pd.DataFrame({f"q={q}, r={r}": {k: v for k, v in c.items() if k not in ("gap_hours_quantiles", "runs_per_utc_hour_of_label")}
                    for (q, r), c in ec["clustering"].items()})
display(clu)
gaps = pd.DataFrame({f"q={q}, r={r}": c.get("gap_hours_quantiles", {}) for (q, r), c in ec["clustering"].items()})
gaps.index.name = "quantile of the gap between consecutive runs (hours)"
gaps

In [ ]:
counts = {k: v["runs"] for k, v in ec["totals"].items()}
chosen = q1.choose(counts, RULE["r_order"], RULE["q_order"], RULE["min_runs"])
print("the rule (Q1_RULE) on the runs at grid labels:", chosen)
for alt in ("runs_noncrossing_240", "runs_noncrossing_60"):
    print(f"  for comparison only, on {alt}:", q1.choose({k: v[alt] for k, v in ec["totals"].items()}, RULE["r_order"], RULE["q_order"], RULE["min_runs"]))
for S in (H["primary"], H["secondary"]):
    print(f"  for comparison only, on runs spaced >= {S} min:",
          q1.choose({k: c[f"greedy_spaced_{S}min"] for k, c in ec["clustering"].items()}, RULE["r_order"], RULE["q_order"], RULE["min_runs"]))
reg_q, reg_r = _REG["Q1_SMALL_Q"], _REG["Q1_RUN_LENGTH"]
print(f"register: Q1_SMALL_Q {reg_q['status']} {reg_q['value']}, Q1_RUN_LENGTH {reg_r['status']} {reg_r['value']}")
if chosen["q"] is not None and reg_q["status"] == "APPROVED-BY-RULE":
    assert (need("Q1_SMALL_Q"), need("Q1_RUN_LENGTH")) == (chosen["q"], chosen["r"]), "the register differs from the rule"
print("outcome analyses so far:", stage1.attempts()["outcome_analyses"])

## Q1 setup: result (counts only; ATTEMPTS.json still 0)

**Labels.** OBSERVED: the cells above (the same as `results/q1_run_counts.json`). INFERRED: my arithmetic or
reasoning. The full write-up is in `Q1_SETUP.md` sections 7–10. Sections 3–6 are the superseded pooled-threshold
round.

**The threshold definition changed before any outcome** (decision log #31): it is now per New York hour, because
the pooled quantile concentrated the runs at 19/20/23/00 UTC.

**The rule's choice.**
- OBSERVED: at r = 4, q = 0.20 gives 349 runs at grid labels in 2012–2020, which is ≥ 300. So the rule chooses
  **Q1_RUN_LENGTH = 4 and Q1_SMALL_Q = 0.20**, APPROVED-BY-RULE (#37).
- OBSERVED: every other reading chooses the same pair.
- Descriptive sensitivity only: q = 0.25, r = 4, with 697 runs.

**Per NY hour.**
- OBSERVED: the runs spread over the day, at most 31 (NY 10). NY 17 has none: it is the daily break.
- OBSERVED: the runs at NY 13–16 (46) all cross the close at H = 240, so 303 runs remain non-crossing at 240 min.

**In the test years 2014–2020** (OBSERVED, `per_year`): 283 runs; 279 non-crossing at 60 min, 245 at 240 min.
2013's 66 runs are training only.

**Approved:** NW lag 8 at both H; Q1_AUG_FIT (pinned β, only γ per fold).

**PROPOSED:** Q1_FOLDS edges and Q1_AUG_ESTIMATOR (`Q1_SETUP.md` section 9).


## Plan (not filled in)

- **Q1 setup:** done above (counts only). `Q1_SETUP.md`: the decisions, the readings, the NW-lag proposal.
- **Stage 2, Q1:** the small-candle pattern against HAR, once Q1_NW_LAG_240, Q1_NW_LAG_60, Q1_FOLDS and
  Q1_AUG_FIT are decided. It is the first outcome analysis, so it increments `ATTEMPTS.json`.
- **Stage 3:** the squeeze definition and event counts.
- **Stage 4:** an order and fill simulator, tested on synthetic data.
- **Stage 5:** PREREG.
- **Stage 6:** one look at untouched data.